# Introduction to Scikit-Learn (sklearn)

This notebook demonstrates some of the most useful functions of the
beautiful Scikit-Learn library.

What we're going to cover:

0. An end-to-end Scikit-Learn workflow
1. Getting the data ready
2. Choose the right estimator (model) / algorithm for our problems
3. Fit the model/algorithm and use it to make predictions on our data
4. Evaluating a model
5. Improve a model
6. Save and load a trained model
7. Putting it all together

## 0. An end-to-end Scikit-Learn workflow

In [ ]:
# 1. Get the data ready
import pandas as pd
import numpy as np
import sklearn

heart_disease = pd.read_csv('./data/heart-disease.csv')
heart_disease

In [ ]:
# Create `X` (the "feature matrix")
# AKA data or feature variables
X = heart_disease.drop('target', axis=1)
X

In [ ]:
# Create the y (AKA labels or label matrix)
y = heart_disease['target']
y

## 2. Choose the right model and hyperparameters

In [ ]:
# Remember, "hyperparameters" are like "dials" we can use to (fine) tune our model
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier()

# We'll keep the default hyperparameters
clf.get_params()

## 3. Fit the model to the training data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
clf.fit(X_train, y_train);

In [ ]:
# Make a (faulty) prediction (using **incorrectly shaped** data)
try:
    y_broken_label = clf.predict(np.array([0, 2, 3, 4]))
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
# Make a (correct) prediction (using **correctly shaped** data)
# Note: we are still on step 3 or our workflow
y_label = clf.predict(X_test)

In [ ]:
y_preds = clf.predict(X_test)
y_preds

In [ ]:
y_test

## 4. Evaluate the model on the training data ...

In [ ]:
clf.score(X_train, y_train)

In [ ]:
# ... and on the test data
clf.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(classification_report(y_test, y_preds))  # clf.predict(X_test)

See the article, [Understanding a Classification Report](https://medium.com/@kohlishivam5522/understanding-a-classification-report-for-your-machine-learning-model-88815e2ce397),
for an explanation of this report.

In [ ]:
# Calculate the "confusion matrix" for test and predicted values
confusion_matrix(y_test, y_preds)

In [ ]:
accuracy_score(y_test, y_preds)

## 5. Improve a model

In [ ]:
# - Generate a one-time 128-bit secret for the seed.
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **Copy and paste** this value as the **hard-coded** random number
# generator seed
rng = np.random.default_rng(seed=77708057215789171477656921825058000355)

In [ ]:
# Try different amount of `n_estimators`
for i in range(10, 100, 10):
    print(f'Trying model with {i} estimators...')
    # I use the `default_rng` with a specified seed to generate reproducible results.
    # The maximum integer I want is the largest unsigned 32-bit integer. (This value
    # is the largest value compatible with the argument to `rng.integers`.)
    clf = RandomForestClassifier(
        n_estimators=i,
        random_state=rng.integers(np.iinfo(np.uint32).max)).fit(X_train, y_train)
    print(f'Model accuracy on test set: {clf.score(X_test, y_test) * 100:.2f}%')
    print('')  # simply to separate runs

The maximum accuracy of the test set, 91.80%, occurs with 50 estimators.

Consequently, we can **improve** our model by using 50 estimators.

## 6. Save and load a trained model

In [ ]:
# We can save a model using `pickle`.
import pickle

In [ ]:
# The video calls `pickle.dump(clf, open('random_forest_model.pkl', 'wb'))`.
# This call results in a type warning similar to
# "Expected SupportsWrite but got BinaryIO".
# This situation is resolved by using `with` below.

with open('./random_forest_model_1.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [ ]:
# What happens if we try to import the saved model.
with open('./random_forest_model_1.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [ ]:
loaded_model.score(X_test, y_test)

This result is the **same** as the last model we tried (90 estimators). **Hooray!**

In [ ]:
# Let's "listify" the contents
what_were_covering = [
    '0. An end-to-end Scikit-Learn workflow',
    '1. Getting the data ready',
    '2. Choose the right estimator/algorithm/model for your problem',
    '3. Fitting your chosen machine learning model to data and using it to make a prediction',
    '4. Evaluating a machine learning model',
    '5. Improving predictions through experimentation (hyperparameter tuning)',
    '6. Saving and loading a pre-trained model',
    '7. Putting it all together in a pipeline',
]

In [ ]:
what_were_covering

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# %matplotlib inline

## 1. Getting our data ready to be used with machine learning

Three main tasks to complete:

1. Split the data into features and labels (usually `X` and `y`)
2. Filling (AKA imputing) or disregarding missing values
3. Converting non-numerical values to numerical values (also called "feature encoding")

In [ ]:
heart_disease.head()

In [ ]:
# The last column is our data to be predicted so we drop it
# Remember that `axis=1` is the **column** axis
# (`axis=0` is the **row** axis)
X = heart_disease.drop('target', axis=1)
X.head()

In [ ]:
y = heart_disease['target']
y.head()

In [ ]:
# Split the features and labels into training and test splits
# We reserve 20% of our data for testing. This amount is "negotiable" for
# different problems.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X.shape

In [ ]:
len(heart_disease)

In [ ]:
len(heart_disease) * 0.8

In [ ]:
242 + 61

In [ ]:
len(heart_disease)

## 1.1 Make sure all data is **numerical**

In [ ]:
car_sales = pd.read_csv('./data/car-sales-extended.csv')
car_sales.head()

In [ ]:
car_sales['Doors'].value_counts()

Notice that `car_sales['Doors']` is **both** numeric **and** categorical.

It is numeric because its values are integers. But it is categorical
because it's (mathematical) range is only a small subset of integers.

As a consequence of the small subset of values, we will treat this column
as a **categorical** column (see our encoding code later).

In [ ]:
len(car_sales)

In [ ]:
car_sales.dtypes

In [ ]:
# Split into `X` and `y`
X = car_sales.drop('Price', axis=1)
y = car_sales['Price']

# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Build machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor() ## Create our model
try:
    model.fit(X_train, y_train) ## Fit our model
    model.score(X_test, y_test) ## Score our model
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')
# Score

Remember, we **must** convert strings (objects) to **numbers**

In [ ]:
# Turn the (object) categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify features by **column names**
categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot',  ## Name our transform
                                  one_hot,  ## The specific transformer
                                  categorical_features)],  ## Applies **only** to `categorical_features`
                                remainder='passthrough')  ## Pass all other columns **unchanged**
transformed_X = transformer.fit_transform(X)
transformed_X

In [ ]:
pd.DataFrame(transformed_X)

In [ ]:
# An alternative to one-hot encoding
dummies = pd.get_dummies(car_sales[['Make', 'Colour', 'Doors']])
dummies

Now that our data is all numeric (zeros and ones), let's refit the model

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=22232115356560702892793270496072236204)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
print(sklearn.__version__)

## 1.2 What if I find **missing** data?

1. Fill them with some value (AKA imputation)
2. Remove the samples with missing data altogether

Neither of these techniques is "perfect" or "recommended"
- Replacing "missing" data might introduce bias or "throw away" "significant" information
- Removing samples completely results in less total data to use

In [ ]:
# Import car sales with missing data
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Extract `X` and `y` (features and values)
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Calling `.sum()` takes advantage of Python "coercion" of Boolean types
# That is, True is converted to 1 when summing and False is converted to 0.
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

At this point in the video, Python reports an exception:
"ValueError: Input contains NaN"

Because I'm using `sklearn` version 1.5.x, I **do not** see this error.

But I'll pretend like I do.

In [ ]:
car_sales_missing

In [ ]:
car_sales_missing['Doors'].value_counts()

In [ ]:
car_sales_missing['Doors'].mode()

#### Option 1: Fill missing data with `pandas`

In [ ]:
# Fill the "Make" column
car_sales_missing['Make'] = car_sales_missing['Make'].fillna('missing')

# Fill the "Colour" column
car_sales_missing['Colour'] = car_sales_missing['Colour'].fillna('missing')

# Fill the "Odometer (KM)" column
car_sales_missing['Odometer (KM)'] = car_sales_missing['Odometer (KM)'].fillna(car_sales_missing['Odometer (KM)'].mean())

# Fill the "Doors" column
# A little tricky because this column is actually a **categorical** column.
# Because the (overwhelming) majority of cars have 4 doors, we will
# replace all missing values in the 'Doors' column with the value 4.
# (Because 4 is the most common value, we could replace the hard-coded
# value of 4 with `car_sales_missing['Doors'].mode()`
car_sales_missing['Doors'] = car_sales_missing['Doors'].fillna(4)

In [ ]:
# Check our `DataFrame` again
car_sales_missing.isna().sum()

In [ ]:
# Because 'Price' is our value column, we **do not** want to replace
# missing values with another value. Instead, we will **remove** all
# rows that are missing a value in the 'Price' column.
car_sales_missing = car_sales_missing.dropna(subset=['Price'])

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
len(car_sales_missing)

In [ ]:
# Remember, one must **always** split data into `X` and `y`
# (features and labels) after **changing** the data.
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

### Option 2. Fill missing values with Scikit-Learn

In [ ]:
# Read data (as usual)
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Check for missing data
car_sales_missing.isna().sum()

In [ ]:
# Remove rows **without** labels
car_sales_missing = car_sales_missing.dropna(subset=['Price'])
car_sales_missing.isna().sum()

In [ ]:
# Split into features and labels
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
X.isna().sum()

In [ ]:
# Fill missing values from Scikit-LearnA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Fill categorical values with 'missing' and numerical values with `mean()`
categorical_imputer = SimpleImputer(strategy='constant', fill_value='missing')
door_imputer = SimpleImputer(strategy='constant', fill_value=4)
numeric_imputer = SimpleImputer(strategy='mean')

# Define columns
categorical_features = ['Make', 'Colour']
door_feature = ['Doors']  ## Because 'Doors' column is a special case
numeric_features = ['Odometer (KM)']

# Create an imputer (that is, something the fills missing data)
imputer = ColumnTransformer([
    ('categorical_features', categorical_imputer, categorical_features),
    ('door_feature', door_imputer, door_feature),
    ('numeric_features', numeric_imputer, numeric_features),
])

# (Finally) Transform the features
filled_X = imputer.fit_transform(X)
filled_X

In [ ]:
# Check our code as we have done previously (using `isna().sum()`
car_sales_missing = pd.DataFrame(filled_X,
                                 columns=['Make', 'Colour', 'Doors', 'Odometer (KM)'])
car_sales_missing

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

Now our data is

- All numeric
- Filled (no missing values)

Let's fit a model!

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=124608693003265431754472593407374562997)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
len(car_sales_missing), len(car_sales)